In [1]:
from bertopic import BERTopic
import json
import time

from nltk.tokenize import word_tokenize
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

In [2]:
K_RANGE = list(range(3, 20))
TOP_N = 10

In [3]:
try:
    with open('../dataProcessed/nurseNotesProcessed.json', 'r') as file:
        nurse_notes = json.load(file)
    print("File loaded successfully.")
    
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")

File loaded successfully.


In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def bertopic_analysis(texts):
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    print(f"Number of texts: {len(texts)}")

    start = time.time() 
    topic_model = BERTopic(
        embedding_model='sentence-transformers/all-MiniLM-L6-v2',
    )
    _topics, _probs = topic_model.fit_transform(texts)
    cluster_topics = list(topic_model.get_topic_info()['Representation'])
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)
        
    print(f"Number of Topics: {len(cluster_topics)}")

In [6]:
all_texts = []
for key in nurse_notes.keys():
    print(f"-----------{key}-----------")
    bertopic_analysis(nurse_notes[key])
    all_texts.extend(nurse_notes[key])

-----------P1-----------
Number of texts: 600


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.6942048630581535
Diversity: 0.46875
Inverse Redundancy: 0.85
Time (seconds): 8.635346174240112
----- Cluster Topics -----
['doctor', 'resident', 'voice', 'need', 'attend', 'med', 'injection', 'concern', 'appear', 'nil']
['night', 'check', 'safety', 'comfortable', 'continue', 'take', 'med', 'settle', 'sleep', 'resident']
['sleep', 'remain', 'self', 'pleasant', 'asleep', 'check', 'care', 'ongoing', 'hourly', 'pleasantly']
['mobilizing', 'relaxed', 'walker', 'staff', 'take', 'content', 'appear', 'chart', 'med', 'assist']
['plan', 'morning', 'adls', 'breakfast', 'report', 'staff', 'room', 'intake', 'care', 'interact']
['restaurant', 'mobile', 'independent', 'attend', 'usual', 'meal', 'need', 'take', 'form', 'med']
['eye', 'care', 'morning', 'choice', 'give', 'drop', 'assist', 'usual', 'day', 'plan']
['toilette', 'ongoing', 'asleep', 'comfortable', 'self', 'check', 'resident', 'toilete', 'peacefully', 'require']
['steroid', 'chart', 'form', 'good', 'give', 'med', 'continue', 'a

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.7744440357165606
Diversity: 0.6666666666666666
Inverse Redundancy: 0.950952380952381
Time (seconds): 3.036504030227661
----- Cluster Topics -----
['care', 'resident', 'chair', 'chart', 'need', 'continue', 'nil', 'assist', 'voice', 'good']
['settle', 'tts', 'comfortably', 'hoist', 'night', 'staff', 'medication', 'give', 'bed', 'drink']
['microlax', 'laxative', 'bno', 'give', 'prn', 'morning', 'remain', 'refuse', 'oral', 'content']
['adls', 'adequate', 'new', 'intake', 'chart', 'nil', 'appear', 'pu', 'concern', 'med']
['vomit', 'vomiting', 'nausea', 'episode', 'feel', 'report', 'urine', 'bell', 'later', 'watch']
['wheelchair', 'baseline', 'electric', 'prescribe', 'wash', 'transfer', 'skin', 'take', 'form', 'appear']
['check', 'comfortable', 'need', 'safety', 'asleep', 'nil', 'med', 'concern', 'care', 'chart']
['night', 'notice', 'check', 'slept', 'continue', 'concern', 'sleep', 'safety', 'resident', 'hourly']
['eye', 'toilet', 'early', 'nocte', 'sit', 'overnight', 'administe

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.7522793334611507
Diversity: 0.5105263157894737
Inverse Redundancy: 0.8929824561403509
Time (seconds): 2.7642149925231934
----- Cluster Topics -----
['bed', 'morning', 'resident', 'care', 'personal', 'med', 'continue', 'give', 'day', 'chart']
['eye', 'good', 'form', 'attend', 'care', 'medication', 'appear', 'voice', 'chart', 'assist']
['adls', 'assisted', 'meds', 'charted', 'compliant', 'maintain', 'night', 'have', 'settle', 'safety']
['time', 'entry', 'concern', 'daughter', 'personal', 'good', 'form', 'med', 'bright', 'attend']
['adls', 'charted', 'compliant', 'assisted', 'meds', 'maintain', 'settle', 'night', 'safety', 'need']
['medication', 'administer', 'alert', 'bright', 'appear', 'new', 'house', 'eating', 'drink', 'concern']
['sensor', 'mat', 'asleep', 'place', 'bed', 'comfortable', 'check', 'issue', 'med', 'appear']
['toilet', 'nocte', 'comfortable', 'early', 'asleep', 'bed', 'check', 'place', 'mat', 'sensor']
['toilete', 'gong', 'place', 'mat', 'sensor', 'assist', '

In [7]:
bertopic_analysis(all_texts)

Number of texts: 12225


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.5836900157820056
Diversity: 0.29884057971014494
Inverse Redundancy: 0.9663431075160094
Time (seconds): 17.386243104934692
----- Cluster Topics -----
['note', 'receive', 'sleep', 'gp', 'keep', 'till', 'med', 'take', 'resident', 'observe']
['laxative', 'bno', 'microlax', 'refuse', 'bowel', 'constipation', 'appeared', 'oral', 'prn', 'soft']
['require', 'activity', 'usual', 'relax', 'visit', 'currently', 'friend', 'form', 'attend', 'enjoy']
['paracetamol', 'pain', 'prn', 'complain', 'hip', 'request', 'leg', 'shoulder', 'gm', 'regular']
['inhaler', 'club', 'social', 'aspiration', 'knitting', 'activity', 'enjoy', 'laxative', 'laxose', 'decline']
['tts', 'hoist', 'usher', 'comfortably', 'pad', 'staff', 'pu', 'clothe', 'drink', 'toilete']
['complaint', 'voice', 'slt', 'medication', 'attend', 'rolator', 'activity', 'nil', 'take', 'form']
['instill', 'eye', 'drop', 'aid', 'appointment', 'eyedrop', 'letter', 'mobilizing', 'provide', 'rollater']
['medicine', 'alert', 'food', 'bright',